# 最大似然估计 (MLE) 示例代码

本文档包含《最大似然估计（MLE）简介》中的三个核心计算示例：
1. **离散型实例**：伯努利分布（抽球问题）
2. **连续型实例**：正态分布（学生身高估计）
3. **进阶对比**：MLE 与 贝叶斯估计（多盒子推断）

## 1. 离散型实例：伯努利分布

**场景**：估计 5 个盒子中抽取出白球的概率。
**数据**：每个盒子抽取 10 次，记录白球（1）和黑球（0）的次数。

In [1]:
import numpy as np
import pandas as pd

# 模拟抽样数据：1 代表白球，0 代表黑球
# 盒子 1-5 的抽样结果（10次）
data_bernoulli = {
    "Box 1": [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    "Box 2": [0, 0, 1, 1, 0, 0, 0, 0, 1, 0],
    "Box 3": [0, 1, 0, 0, 1, 0, 1, 0, 1, 1],
    "Box 4": [1, 0, 1, 1, 1, 0, 1, 0, 1, 1],
    "Box 5": [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
}

df_bernoulli = pd.DataFrame(data_bernoulli)
print("抽样数据概览：")
print(df_bernoulli)

# MLE 计算：p_hat = sum(x_i) / n
print("\n--- MLE 估计结果 ---")
for box in df_bernoulli.columns:
    # 计算白球次数
    white_count = df_bernoulli[box].sum()
    n = len(df_bernoulli[box])
    
    # 计算 MLE
    p_hat = white_count / n
    print(f"{box}: 白球次数 = {white_count}, n = {n}, p_hat = {p_hat:.1f}")

抽样数据概览：
   Box 1  Box 2  Box 3  Box 4  Box 5
0      0      0      0      1      1
1      0      0      1      0      1
2      0      1      0      1      1
3      0      1      0      1      1
4      0      0      1      1      1
5      0      0      0      0      1
6      0      0      1      1      1
7      0      0      0      0      1
8      0      1      1      1      1
9      0      0      1      1      1

--- MLE 估计结果 ---
Box 1: 白球次数 = 0, n = 10, p_hat = 0.0
Box 2: 白球次数 = 3, n = 10, p_hat = 0.3
Box 3: 白球次数 = 5, n = 10, p_hat = 0.5
Box 4: 白球次数 = 7, n = 10, p_hat = 0.7
Box 5: 白球次数 = 10, n = 10, p_hat = 1.0


## 2. 连续型实例：正态分布

**场景**：估计班级学生的平均身高和方差。
**数据**：5 名学生的身高 [170, 172, 168, 175, 165] (cm)。

In [2]:
# 样本数据
heights = np.array([170, 172, 168, 175, 165])
n = len(heights)

print(f"样本数据: {heights}")
print(f"样本量 n: {n}")

# 1. 求解均值 mu_hat
mu_hat = np.mean(heights)
print(f"\n均值 MLE (mu_hat): {mu_hat} cm")

# 2. 求解方差 sigma_sq_hat (MLE)
# 注意：MLE 的方差分母是 n，而 pandas/numpy 默认 var 的 ddof=0 即为分母 n
# ddof=1 为无偏估计 (分母 n-1)
sigma_sq_mle = np.var(heights, ddof=0)
print(f"方差 MLE (sigma^2_hat): {sigma_sq_mle} cm^2")

# 补充：无偏方差估计
sigma_sq_unbiased = np.var(heights, ddof=1)
print(f"(参考) 无偏样本方差: {sigma_sq_unbiased} cm^2")

样本数据: [170 172 168 175 165]
样本量 n: 5

均值 MLE (mu_hat): 170.0 cm
方差 MLE (sigma^2_hat): 11.6 cm^2
(参考) 无偏样本方差: 14.5 cm^2


## 3. 进阶示例：MLE 与 贝叶斯估计对比

**场景**：5 个盒子，白球比例已知。有放回抽取 2 个球均为白球，推断来自哪个盒子。

**已知条件**：
- 盒子白球比例 $p_i$: [0, 0.3, 0.5, 0.7, 1.0]
- 观测事件 $B$: 两次均为白球

In [3]:
# 定义盒子及其白球比例
boxes = ["Box 1", "Box 2", "Box 3", "Box 4", "Box 5"]
p_white = np.array([0, 0.3, 0.5, 0.7, 1.0])

# 观测事件：两次均为白球
# P(B | Box_i) = p_i^2
likelihoods = p_white ** 2

print("--- 1. 最大似然估计 (MLE) ---")
df_mle = pd.DataFrame({
    "Box": boxes,
    "Likelihood (p^2)": likelihoods
})
print(df_mle)

best_box_mle = boxes[np.argmax(likelihoods)]
max_val_mle = np.max(likelihoods)
print(f"\nMLE 结论: 最有可能来自 {best_box_mle} (似然值 = {max_val_mle:.2f})")


print("\n--- 2. 贝叶斯估计 (Bayesian) ---")
# 先验概率：假设均匀分布
priors = np.array([1/5] * 5)

# 计算边缘概率 P(B) = sum(P(B|A_i) * P(A_i))
p_b = np.sum(likelihoods * priors)
print(f"总概率 P(B): {p_b:.3f}")

# 计算后验概率 P(A_i | B) = (P(B|A_i) * P(A_i)) / P(B)
posteriors = (likelihoods * priors) / p_b

df_bayes = pd.DataFrame({
    "Box": boxes,
    "Prior": priors,
    "Likelihood": likelihoods,
    "Posterior": posteriors
})
print(df_bayes)

best_box_bayes = boxes[np.argmax(posteriors)]
max_val_bayes = np.max(posteriors)
print(f"\n贝叶斯结论: 最有可能来自 {best_box_bayes} (后验概率 = {max_val_bayes:.3f} or {max_val_bayes*100:.1f}%)")

--- 1. 最大似然估计 (MLE) ---
     Box  Likelihood (p^2)
0  Box 1              0.00
1  Box 2              0.09
2  Box 3              0.25
3  Box 4              0.49
4  Box 5              1.00

MLE 结论: 最有可能来自 Box 5 (似然值 = 1.00)

--- 2. 贝叶斯估计 (Bayesian) ---
总概率 P(B): 0.366
     Box  Prior  Likelihood  Posterior
0  Box 1    0.2        0.00   0.000000
1  Box 2    0.2        0.09   0.049180
2  Box 3    0.2        0.25   0.136612
3  Box 4    0.2        0.49   0.267760
4  Box 5    0.2        1.00   0.546448

贝叶斯结论: 最有可能来自 Box 5 (后验概率 = 0.546 or 54.6%)
